# Real-Time Order Book Alpha Research Dashboard
---
This notebook demonstrates the loading and visualization of engineered alpha features and model performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.style.use('dark_background')
sns.set_theme(style="darkgrid", palette="pastel")

# Update path if running outside the project root
import sys
sys.path.append('..')
from src.config import STORAGE

### Load Processed Features

In [ ]:
features_path = os.path.join(STORAGE.PROCESSED_DIR, "features.parquet")
if os.path.exists(features_path):
    df_features = pd.read_parquet(features_path)
    display(df_features.head())
    
    plt.figure(figsize=(14, 4))
    plt.plot(pd.to_datetime(df_features['timestamp'], unit='ms'), df_features['mid_price'])
    plt.title('Mid Price Over Time')
    plt.show()
else:
    print("Features not found. Run the ingestion and feature engineering pipelines first.")

### Feature Correlations

In [ ]:
if 'df_features' in locals():
    corr = df_features[['spread', 'depth_imbalance', 'volatility_10', 'volatility_50', 'ofi', 'returns']].corr()
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Feature Correlation Matrix")
    plt.show()

### Analyze Alpha Model Predictions & Backtest Performance

In [ ]:
pred_path = os.path.join(STORAGE.PROCESSED_DIR, "predictions.parquet")
if os.path.exists(pred_path):
    df_preds = pd.read_parquet(pred_path)
    
    # Scatter Plot of Actual Return vs Predicted Return
    plt.figure(figsize=(8, 6))
    plt.scatter(df_preds['predicted_return'], df_preds['actual_return'], alpha=0.5)
    plt.xlabel('Predicted Return')
    plt.ylabel('Actual Return')
    plt.title('Prediction vs Actual')
    # Add a reference line
    min_val = min(df_preds['predicted_return'].min(), df_preds['actual_return'].min())
    max_val = max(df_preds['predicted_return'].max(), df_preds['actual_return'].max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--')
    plt.show()
else:
    print("Predictions not found. Run the alpha_model pipeline first.")